<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aibizx/learn-ml-algorithms-2017-notebooks/blob/main/11-clustering.ipynb)

_Part of the [AI/Biz books](https://www.ai.biz/books/learn-ml-algorithms-2017/) collection._

# Chapter 11 — Clustering

Companion to [the chapter](https://www.ai.biz/books/learn-ml-algorithms-2017/clustering/).

Finding groups with no answer key, and the harder question that follows: whether the groups are real.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons

# Customers described by two standardised numbers: how often they buy, and how much per order
X, true_group = make_blobs(n_samples=600, centers=4, cluster_std=0.9, random_state=7)
print(f'{len(X)} customers, 2 features. We happen to know there are 4 groups; the algorithm will not.')

## 1. k-means, from scratch

In [ ]:
def kmeans(X, k, seed=0, max_iter=100):
    rng = np.random.default_rng(seed)
    centres = X[rng.choice(len(X), k, replace=False)]          # start from k random customers
    for it in range(max_iter):
        d = np.linalg.norm(X[:, None, :] - centres[None, :, :], axis=2)   # every customer to every centre
        labels = d.argmin(axis=1)                                          # assign
        new = np.array([X[labels == j].mean(axis=0) if np.any(labels == j) else centres[j] for j in range(k)])
        if np.allclose(new, centres):
            break
        centres = new                                                      # update
    inertia = np.sum((X - centres[labels]) ** 2)
    return labels, centres, inertia, it + 1

labels, centres, inertia, iters = kmeans(X, 4)
print(f'converged in {iters} iterations; inertia {inertia:.1f}')
print('centres:\n', np.round(centres, 2))

## 2. Different starts, different answers

In [ ]:
results = [kmeans(X, 4, seed=s) for s in range(20)]
inertias = np.array([r[2] for r in results])
print(f'20 random starts: inertia from {inertias.min():.1f} to {inertias.max():.1f}')
print(f'{np.sum(inertias > inertias.min() * 1.05)} of 20 starts ended in a worse local minimum')

## 3. The one line you would ship

In [ ]:
from sklearn.cluster import KMeans

km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(X)
print(f'KMeans with k-means++ starts and the best of 10 runs: inertia {km.inertia_:.1f}')

## 4. Choosing k: the elbow and the silhouette

In [ ]:
from sklearn.metrics import silhouette_score

print(f'{"k":>3} {"inertia":>9} {"silhouette":>11}')
for k in range(2, 9):
    m = KMeans(n_clusters=k, n_init=10, random_state=0).fit(X)
    print(f'{k:>3} {m.inertia_:>9.1f} {silhouette_score(X, m.labels_):>11.3f}')

## 5. The same tools on data with no groups in it

In [ ]:
rng = np.random.default_rng(0)
U = rng.normal(0, 5, (600, 2))                                # one cloud of customers, no groups in it
print(f'{"k":>3} {"inertia":>9} {"silhouette":>11}')
for k in range(2, 7):
    m = KMeans(n_clusters=k, n_init=10, random_state=0).fit(U)
    print(f'{k:>3} {m.inertia_:>9.0f} {silhouette_score(U, m.labels_):>11.3f}')

## 6. Is the cluster real? Compare against nothing

In [ ]:
def gap_like(X, k, trials=10, seed=0):
    """Inertia on the data, against inertia on uniform noise filling the same box. Larger ratio, realer structure."""
    rng = np.random.default_rng(seed)
    real = KMeans(n_clusters=k, n_init=5, random_state=0).fit(X).inertia_
    lo, hi = X.min(axis=0), X.max(axis=0)
    fake = [KMeans(n_clusters=k, n_init=5, random_state=t).fit(rng.uniform(lo, hi, X.shape)).inertia_ for t in range(trials)]
    return np.log(np.mean(fake)) - np.log(real)

print(f'{"k":>3} {"gap, clustered":>15} {"gap, one cloud":>15}')
for k in range(1, 7):
    print(f'{k:>3} {gap_like(X, k):>15.3f} {gap_like(U, k):>15.3f}')

## 7. Stability: do the groups survive a different sample?

In [ ]:
from sklearn.metrics import adjusted_rand_score

def stability(X, k, trials=10, seed=0):
    rng = np.random.default_rng(seed)
    scores = []
    for t in range(trials):
        a = rng.choice(len(X), len(X) // 2, replace=False)
        b = rng.choice(len(X), len(X) // 2, replace=False)
        ma = KMeans(n_clusters=k, n_init=5, random_state=t).fit(X[a])
        mb = KMeans(n_clusters=k, n_init=5, random_state=t).fit(X[b])
        both = np.intersect1d(a, b)
        scores.append(adjusted_rand_score(ma.predict(X[both]), mb.predict(X[both])))
    return np.mean(scores)

print(f'{"k":>3} {"agreement, clustered":>21} {"agreement, one cloud":>21}')
for k in [2, 3, 4, 5, 6]:
    print(f'{k:>3} {stability(X, k):>21.2f} {stability(U, k):>21.2f}')

## 8. Scale decides what "near" means

In [ ]:
spend = np.column_stack([rng.uniform(1, 6, 600), rng.uniform(500, 50000, 600)])    # orders per year, rupees per year
spend[:300, 0] += 14                                                             # half the customers order far more often
from sklearn.preprocessing import StandardScaler

raw = KMeans(n_clusters=2, n_init=10, random_state=0).fit(spend)
scaled = KMeans(n_clusters=2, n_init=10, random_state=0).fit(StandardScaler().fit_transform(spend))
frequent = np.arange(600) < 300
print(f'raw features   : clusters split frequent from infrequent buyers {adjusted_rand_score(frequent, raw.labels_):.2f} of the way')
print(f'scaled features: {adjusted_rand_score(frequent, scaled.labels_):.2f}')
print('raw centres (orders, rupees):', np.round(raw.cluster_centers_, 0))

## 9. Shapes k-means cannot see

In [ ]:
from sklearn.cluster import DBSCAN, AgglomerativeClustering

Xm, ym = make_moons(400, noise=0.07, random_state=0)
for name, m in [('k-means, k=2', KMeans(n_clusters=2, n_init=10, random_state=0)),
                ('agglomerative, single linkage', AgglomerativeClustering(n_clusters=2, linkage='single')),
                ('DBSCAN', DBSCAN(eps=0.2, min_samples=5))]:
    found = m.fit_predict(Xm)
    print(f'{name:<30} agreement with the two moons {adjusted_rand_score(ym, found):.2f}   groups found {len(set(found) - {-1})}')

## 10. DBSCAN: density, and the points that belong nowhere

In [ ]:
Xn = np.vstack([Xm, rng.uniform(-1.5, 2.5, (30, 2))])        # thirty stray points
print(f'{"eps":>5} {"groups":>7} {"noise points":>13}')
for eps in [0.05, 0.1, 0.2, 0.4]:
    found = DBSCAN(eps=eps, min_samples=5).fit_predict(Xn)
    print(f'{eps:>5} {len(set(found) - {-1}):>7} {np.sum(found == -1):>13}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, (name, found) in zip(axes, [('k-means', KMeans(2, n_init=10, random_state=0).fit_predict(Xn)),
                                     ('single linkage', AgglomerativeClustering(2, linkage='single').fit_predict(Xn)),
                                     ('DBSCAN eps=0.2', DBSCAN(eps=0.2, min_samples=5).fit_predict(Xn))]):
    ax.scatter(Xn[:, 0], Xn[:, 1], c=found, cmap='tab10', s=12)
    ax.set_title(name)
plt.tight_layout(); plt.show()

## 11. Hierarchical clustering: the tree of merges

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster

Z = linkage(X, method='ward')
print('the last five merges (distance at which they happened):', np.round(Z[-5:, 2], 1))
for k in [2, 3, 4, 5]:
    cut = fcluster(Z, k, criterion='maxclust')
    print(f'cut into {k}: agreement with the four true groups {adjusted_rand_score(true_group, cut):.2f}')

## 12. A cluster as an anomaly detector

In [ ]:
d = np.linalg.norm(X - km.cluster_centers_[km.labels_], axis=1)
threshold = np.percentile(d, 99)
odd = np.where(d > threshold)[0]
print(f'99th percentile of distance to own centre: {threshold:.2f}; {len(odd)} customers beyond it')
new_customer = np.array([[8.0, 8.0]])
print(f'a new customer at {new_customer[0]}: distance to nearest centre {np.min(np.linalg.norm(km.cluster_centers_ - new_customer, axis=1)):.2f}')

## Try it yourself

1. In section 4, set `cluster_std=2.5` when making the blobs. Can the elbow or the silhouette still find 4?
2. In section 7, what agreement does k=4 get on the uniform data if you use 90% of the rows for each half instead of 50%? Why is that a worse test?
3. In section 10, set `min_samples=20`. What happens to the moons?